In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader
from pathlib import Path
import os


c:\Users\Dillon\anaconda3\envs\dl\Lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [ ]:
# Load Model
def create_model(type):
    if type == 'CNN':
        model = models.resnet50(weights="DEFAULT")
        model.fc = nn.Linear(2048, 1)
        
    elif type == 'ViT':
        model = models.vit_b_16(weights = None)
        model.heads.head = nn.Sequential(
        nn.ReLU(), 
        nn.Linear(768, 1),  # Binary classification
        )
    
    return model

# Load the model
model = create_model('ViT')
model.load_state_dict(torch.load("Enter path to model weight .pth file", weights_only=True))


<All keys matched successfully>

In [ ]:
# Transform Dataset
test_path = Path("Enter path to testset")

test_transformer = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
# Conversion of files to PyTorch Dataset and DataLoader
test_data = datasets.ImageFolder(root=test_path, transform=test_transformer)

test_loader = DataLoader(test_data, batch_size=1, shuffle=False, num_workers=4)

In [4]:
# Test counting correct fp and fn
# Since trained as binary classifier, change dataset from GAN to Diffusion or vice verse
#Real Dataset should remain the same

model.eval()
    
tp = 0
tn = 0
fp = 0
fn = 0

with torch.inference_mode():
    for i, inp in enumerate(test_loader):
        inputs, labels = inp
        inputs, labels = inputs, labels

        outputs = model(inputs).squeeze()
        preds = torch.round(torch.sigmoid(outputs))
        
        
        tp += ((preds.view(-1) == 0) & (labels == 0)).sum().item()
        tn += ((preds.view(-1) == 1) & (labels == 1)).sum().item()
        fp += ((preds.view(-1) == 1) & (labels == 0)).sum().item()
        fn += ((preds.view(-1) == 0) & (labels == 1)).sum().item()
        
print(f"True Positives: {tp}")
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Positives: {fn}")

True Positives: 445
True Negatives: 456
False Positives: 55
False Positives: 44
